# 04 Sensitivity

重み `w1, w2, w3` を変えたとき、地点の `Ji` 順位がどう動くかを見る。

- プリセット比較（balanced / elderly / commuter / heat_alert）
- 単項除去（各項をゼロにしたときの影響）


In [1]:
from __future__ import annotations
from pathlib import Path
import sys

import pandas as pd
import yaml

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from heat_town.model import compute_ji, normalize_weights  # noqa: E402


In [2]:
# features.parquet を読む。無ければ pipeline --sample を促す
features_path = ROOT / 'data' / 'processed' / 'features.parquet'
if not features_path.exists():
    raise FileNotFoundError('features.parquet がありません。`python -m heat_town.cli pipeline --sample` を先に実行してください。')
df = pd.read_parquet(features_path)
df.head()


,grid_id,latitude,longitude,d,C,WBGT,hour
0,g0000,35.625466,139.779500,0.8947,37.86,24.6,15
1,g0001,35.625466,139.780605,0.8489,47.94,24.6,15
2,g0002,35.625466,139.781711,0.8060,57.62,24.6,15
3,g0003,35.625466,139.782816,0.7665,70.97,24.6,15
4,g0004,35.625466,139.783921,0.7310,70.92,24.6,15


In [3]:
# config のプリセットを読む
with open(ROOT / 'config' / 'weights.yaml', encoding='utf-8') as f:
    presets = yaml.safe_load(f)['presets']
presets


{'balanced': {'w1': 0.3, 'w2': 0.4, 'w3': 0.3},
 'elderly': {'w1': 0.2, 'w2': 0.5, 'w3': 0.3},
 'commuter': {'w1': 0.5, 'w2': 0.2, 'w3': 0.3},
 'heat_alert': {'w1': 0.2, 'w2': 0.2, 'w3': 0.6}}

In [4]:
# 各プリセットで Ji を計算し、順位列を作る
ranks = pd.DataFrame({'grid_id': df['grid_id']})
for name, w in presets.items():
    n1, n2, n3 = normalize_weights(w['w1'], w['w2'], w['w3'])
    ji = n1 * df['d'] + n2 * (100 - df['C']) + n3 * df['WBGT']
    ranks[name] = ji.rank(method='min').astype(int)
ranks.head()


,grid_id,balanced,elderly,commuter,heat_alert
0,g0000,355,355,357,355
1,g0001,280,278,289,282
2,g0002,182,180,185,184
3,g0003,38,37,43,39
4,g0004,39,38,41,40


## プリセット間の順位変動

`balanced` を基準に、各プリセットで順位が最も動いた地点を見る。

In [5]:
base = 'balanced'
for name in presets:
    if name == base:
        continue
    delta = (ranks[name] - ranks[base]).abs()
    top = delta.sort_values(ascending=False).head(3)
    print(f'{base} -> {name}: 最大順位変動 {int(delta.max())} 位 (地点 {list(top.index)})')


balanced -> elderly: 最大順位変動 5 位 (地点 [398, 189, 250])
balanced -> commuter: 最大順位変動 12 位 (地点 [188, 211, 209])
balanced -> heat_alert: 最大順位変動 3 位 (地点 [109, 160, 211])


## まとめ

- 高齢者プリセット（w2↑）で不快度の高い地点が相対的に不利になる
- 猛暑日プリセット（w3↑）で WBGT の高い地点の順位が上がる
- → 重み設定が優先順位を実際に変えることを確認（説明可能性）
